# A New Branch-and-Bound Pruning Framework for L0-Regularized Problems

*T. Guyard, C. Elvira, C. Herzet, and A.-N. Arslan. ICML 2024.*

#### Notebook to reproduce experiments of Section 4.2

---

This experiment compares the performance of different methods to fit a regularization path for on L0-regularized problems with real-worlds data. To reproduce it, let first import all the necessary packages and routines.

In [ ]:
from el0ps.path import Path
from el0ps.utils import compute_lmbd_max
from l0exp.calibration import get_calibration_l0learn
from l0exp.dataset import get_dataset_realworld
from l0exp.solver import get_solver, can_handle_instance

Now, let select the dataset to use. Each dataset contains a feature matrix `A` and a target vector `y`. Possible choices are:
- `riboflavin`: Dataset to characterize riboflavin production of Bacillus subtilis cells depending on gene expression.
- `bctcga`: **[Currently broken, to be fixed soon]** Breast cancer gene expression dataset from The Cancer Genome Atlas.
- `colon-cancer`: Cancer screening dataset from LIBSVM.
- `leukemia`: Cancer screening dataset from LIBSVM.
- `arcene`: Cancer screening dataset from the NIPS 2003 feature selection challenge.
- `breast-cancer`: Cancer screening dataset from LIBSVM (`duke breast-cancer`).
By default, datasets are normalized.

In [ ]:
dataset = "leukemia"
A, y, x = get_dataset_realworld(name=dataset, normalize=True)
print(f"Dataset: {dataset}")
print(f"A shape: {A.shape}")
print(f"y shape: {y.shape}")

We now use the `L0Learn` package to calibrate appropriate data-fidelity and penalty functions, as well as the value of the regularization parameter `lmbd`. In our experiments, we used the following data-fidelity functions for each dataset:

| Dataset         | Data-fidelity function  |
|-----------------|-------------------------|
| `riboflavin`    | `Leastsquares`          |
| `bctcga`        | `Leastsquares`          |
| `colon-cancer`  | `Logistic`              |
| `leukemia`      | `Logistic`              |
| `arcene`        | `Squaredhinge`          |
| `breast-cancer` | `Squaredhinge`          |

Both penalties `BigmL1norm` and `BigmL2norm` have also been considered.

In [ ]:
datafit, penalty, lmbd = get_calibration_l0learn(A, y, x, datafit="Logistic", penalty="BigmL2norm")
print(f"datafit   : {datafit}")
print(f"penalty   : {penalty}")
print(f"lambda    : {lmbd}")
print(f"lambda_max: {compute_lmbd_max(datafit, penalty, A)}")

Finally, we can define the different methods to compare and their parameters. Here is the setup used in our experiments. Solvers that cannot handle the considered instance are automatically skipped.

In [ ]:
# Solvers to use (comment out those you don't want to run or that are not installed)
solver_types = [
    "el0ps",
    "l0bnb",
    "mimosa",
    "gurobi",
    "mosek",
    "oa"
]

solver_args = {
    "time_limit"    : 3600,   # time limit in seconds for each value of lambda in the path
    "relative_gap"  : 1.e-8,  # relative optimality gap on the objective value
    "verbose"       : False,  # verbosity toggle for solvers
}

path_args = {
    "lmbd_max"          : 1.0,      # maximum value of lmbd / lmbd_max in the path
    "lmbd_min"          : 0.01,     # minimum value of lmbd / lmbd_max in the path
    "lmbd_num"          : 20,       # number of values of lmbd in the path
    "lmbd_normalized"   : True,     # normalize lmbd w.r.t lmbd_max in the path
}

for solver_type in solver_types:

    try:    
        solver = get_solver(solver_type, solver_args)
        if can_handle_instance(solver, datafit, penalty):
            print(f"Running {solver_type}...")
            path = Path(**path_args)
            path.fit(solver, datafit, penalty, A)
        else:
            print(f"Skipping {solver_type}: cannot handle this instance")
    except Exception as e:
        print(f"Solver {solver_type} failed with error: {e}")
    print()